In [1]:
import pandas as pd
import requests
import io
import os
import difflib

In [2]:
file_path = 'data/deteksi_slang_dengan_konteks.csv'
df = pd.read_csv(file_path)

In [3]:
url = 'https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv'
response = requests.get(url)
df_alay = pd.read_csv(io.StringIO(response.text))

In [4]:
dict_alay = dict(zip(df_alay['slang'], df_alay['formal']))
    
# Buat list kata baku unik untuk target Fuzzy Matching (Layer 2)
daftar_kata_baku = list(set(df_alay['formal'].dropna()))

# 3. Mulai Proses Hibrida
hasil_baku = []
metode_deteksi = []

In [5]:
for kata in df['Singkatan/Slang']:
  kata = str(kata).lower()
  
  # LAYER 1: Lexicon Exact Match (Pasti benar)
  if kata in dict_alay:
      hasil_baku.append(dict_alay[kata])
      metode_deteksi.append("1. Lexicon (Akurat)")
      
  else:
      # LAYER 2: Fuzzy Matching (Mencari Typo)
      # cutoff=0.85 berarti kemiripan huruf minimal 85% 
      mirip = difflib.get_close_matches(kata, daftar_kata_baku, n=1, cutoff=0.85)
      
      if mirip:
          hasil_baku.append(mirip[0])
          metode_deteksi.append("2. Fuzzy Match (Typo)")
      else:
          # LAYER 3: Singkatan Ekstrem (Lempar ke Tim buat dicek bareng Konteksnya)
          hasil_baku.append("")
          metode_deteksi.append("3. Manual (Butuh Review)")

In [6]:
df['Kata Baku (Auto/Manual)'] = hasil_baku
df['Metode Deteksi'] = metode_deteksi

# Kita rapikan urutan kolomnya biar enak dilihat di Excel
cols = ['Singkatan/Slang', 'Frekuensi', 'Kata Baku (Auto/Manual)', 'Metode Deteksi', 
        'Contoh Ulasan 1', 'Contoh Ulasan 2', 'Contoh Ulasan 3']
df = df[cols]

# Tampilkan Statistik
total_lexicon = len(df[df['Metode Deteksi'] == '1. Lexicon (Akurat)'])
total_fuzzy = len(df[df['Metode Deteksi'] == '2. Fuzzy Match (Typo)'])
total_manual = len(df[df['Metode Deteksi'] == '3. Manual (Butuh Review)'])

print(f"- Berhasil diisi oleh Lexicon    : {total_lexicon} kata")
print(f"- Berhasil ditebak Typo-nya      : {total_fuzzy} kata")
print(f"- Sisa yang harus dicek Manual   : {total_manual} kata")

- Berhasil diisi oleh Lexicon    : 1900 kata
- Berhasil ditebak Typo-nya      : 0 kata
- Sisa yang harus dicek Manual   : 0 kata


In [7]:
display(df.head(15))

,Singkatan/Slang,Frekuensi,Kata Baku (Auto/Manual),Metode Deteksi,Contoh Ulasan 1,Contoh Ulasan 2,Contoh Ulasan 3
0,gak,23883,enggak,1. Lexicon (Akurat),[BCAMobile] masa gue isi flazz selalu gagal ke...,[BCAMobile] setelah masuk mbanking aktivasi ul...,[BCAMobile] payah jauh sama livin mandiri kiri...
1,ga,20600,enggak,1. Lexicon (Akurat),[BCAMobile] qris ga di pakai daritadi loading ...,[BCAMobile] d update malah minta masukin norek...,[BCAMobile] akun nya ga bis akembali
2,aja,18153,saja,1. Lexicon (Akurat),[BCAMobile] ribet banget ni apk sumpah dikir v...,[BCAMobile] payah jauh sama livin mandiri kiri...,[BCAMobile] apk dongo login aja kaga
3,udah,15884,sudah,1. Lexicon (Akurat),[BCAMobile] tiap verifikasi wajah knapa slalu ...,[BCAMobile] indikator lampu merah terus mana s...,[BCAMobile] verifikasi e ktp wajah selalu gaga...
4,yg,14415,yang,1. Lexicon (Akurat),[BCAMobile] setelah masuk mbanking aktivasi ul...,[BCAMobile] udah restart udah x masuk qris buf...,[BCAMobile] yg terupdate tiap login mesti x si...
5,bca,10570,baca,1. Lexicon (Akurat),[BCAMobile] kasik bintang dulu aku new user bc...,[BCAMobile] bca lgi gangguan apa gimana loadin...,[BCAMobile] tiap verifikasi wajah knapa slalu ...
6,gk,9925,enggak,1. Lexicon (Akurat),[BCAMobile] sekarang bca mobile gk di akses sy...,[BCAMobile] gk tahu eror mulu,[BCAMobile] nomor udah benar kemarin mau pinda...
7,pake,8049,pakai,1. Lexicon (Akurat),[BCAMobile] ribet banget ni apk sumpah dikir v...,[BCAMobile] aplikasi nya sampah tiba tiba kelu...,[BCAMobile] yaa tiap mau buat m bca selalu stu...
8,gimana,7320,bagaimana,1. Lexicon (Akurat),[BCAMobile] bca lgi gangguan apa gimana loadin...,[BCAMobile] kok nggk nerima kode otp nya sejam...,[BCAMobile] gimana tolong habis ganti hp baru ...
9,minta,5501,meminta,1. Lexicon (Akurat),[BCAMobile] d update malah minta masukin norek...,[BCAMobile] kenapa sama mau masuk aplikasi mau...,[BCAMobile] dipindahkan hp laen diulang dr awa...


In [8]:
output_name = 'data/dictionary + konteks.csv'
df.to_csv(output_name, index=False)